In [1]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [2]:
datasetPath = Path("/home/philipm/Desktop/ML-For-CV-Robustness/datasets/notrees.csv")

#making sure
datasetPath.is_file()

True

In [3]:
dataset = pd.read_csv(datasetPath)
dataset.head()

,grviIn,meanRefR,meanRefG,illumR,illumG,blockIdx,sunRoll,sunPitch,sunYaw,camRoll,camPitch,camYaw,grviOut
0,0.163549,0.000324,0.000421,1130.973355,1020.232214,0.0,180.0,-63.812729,-56.07412,0.0,-19.258,-0.0,0.088270
1,0.097110,0.000412,0.000486,1130.973355,1020.232214,1.0,180.0,-63.812729,-56.07412,0.0,-19.258,-0.0,0.057202
2,0.074345,0.000448,0.000517,1130.973355,1020.232214,2.0,180.0,-63.812729,-56.07412,0.0,-19.258,-0.0,0.048503
3,0.156574,0.000374,0.000481,1130.973355,1020.232214,3.0,180.0,-63.812729,-56.07412,0.0,-19.258,-0.0,0.086477
4,0.116627,0.000393,0.000479,1130.973355,1020.232214,4.0,180.0,-63.812729,-56.07412,0.0,-19.258,-0.0,0.067095


In [4]:
target_col = dataset["grviOut"]
labels = target_col.to_numpy(dtype="float32")
labels

array([0.08826979, 0.05720193, 0.04850335, ..., 0.09717431, 0.09026365,
       0.10635003], shape=(333480,), dtype=float32)

In [5]:
dataset = dataset.drop("grviOut", axis=1)
dataset.head()

,grviIn,meanRefR,meanRefG,illumR,illumG,blockIdx,sunRoll,sunPitch,sunYaw,camRoll,camPitch,camYaw
0,0.163549,0.000324,0.000421,1130.973355,1020.232214,0.0,180.0,-63.812729,-56.07412,0.0,-19.258,-0.0
1,0.097110,0.000412,0.000486,1130.973355,1020.232214,1.0,180.0,-63.812729,-56.07412,0.0,-19.258,-0.0
2,0.074345,0.000448,0.000517,1130.973355,1020.232214,2.0,180.0,-63.812729,-56.07412,0.0,-19.258,-0.0
3,0.156574,0.000374,0.000481,1130.973355,1020.232214,3.0,180.0,-63.812729,-56.07412,0.0,-19.258,-0.0
4,0.116627,0.000393,0.000479,1130.973355,1020.232214,4.0,180.0,-63.812729,-56.07412,0.0,-19.258,-0.0


In [6]:
inputs = dataset.drop(["sunRoll", "sunPitch", "sunYaw", "camRoll", "camPitch", "camYaw"], axis=1)
inputs.head()

,grviIn,meanRefR,meanRefG,illumR,illumG,blockIdx
0,0.163549,0.000324,0.000421,1130.973355,1020.232214,0.0
1,0.097110,0.000412,0.000486,1130.973355,1020.232214,1.0
2,0.074345,0.000448,0.000517,1130.973355,1020.232214,2.0
3,0.156574,0.000374,0.000481,1130.973355,1020.232214,3.0
4,0.116627,0.000393,0.000479,1130.973355,1020.232214,4.0


In [7]:
inputs = inputs.to_numpy(dtype="float32")
inputs

array([[1.63549468e-01, 3.24192282e-04, 4.20509488e-04, 1.13097339e+03,
        1.02023224e+03, 0.00000000e+00],
       [9.71098840e-02, 4.11663583e-04, 4.86339268e-04, 1.13097339e+03,
        1.02023224e+03, 1.00000000e+00],
       [7.43446946e-02, 4.47855098e-04, 5.17319597e-04, 1.13097339e+03,
        1.02023224e+03, 2.00000000e+00],
       ...,
       [2.84331501e-01, 2.70231743e-04, 4.24683909e-04, 5.50956787e+02,
        4.80270966e+02, 1.70000000e+01],
       [2.41989389e-01, 3.46322980e-04, 5.10736078e-04, 5.50956787e+02,
        4.80270966e+02, 1.80000000e+01],
       [2.75648892e-01, 3.27728281e-04, 5.15148800e-04, 5.50956787e+02,
        4.80270966e+02, 1.90000000e+01]], shape=(333480, 6), dtype=float32)

In [8]:
if len(labels) != len (inputs):
    print ("problem")

In [9]:
n_total = len(inputs)

n_train = int(0.8*n_total)
n_test = int(0.1*n_total)
n_val = int(0.1*n_total)

# shuffle
g = torch.Generator().manual_seed(42)
perm = torch.randperm(n_total, generator=g).numpy()

inputs_shuf = inputs[perm]
labels_shuf = labels[perm]

# split
train_inputs, train_labels = inputs_shuf[:n_train], labels_shuf[:n_train]
val_inputs, val_labels = inputs_shuf[n_train:n_train + n_val], labels_shuf[n_train:n_train + n_val]
test_inputs, test_labels = inputs_shuf[n_train + n_val:], labels_shuf[n_train + n_val:]

In [10]:
# scaling
from sklearn.preprocessing import QuantileTransformer, StandardScaler

if True:
    input_scaler = StandardScaler() #QuantileTransformer(output_distribution='normal')
    target_scaler = StandardScaler()

    train_inputs = input_scaler.fit_transform(train_inputs)
    train_labels = target_scaler.fit_transform(train_labels.reshape(-1, 1)).reshape(-1)

    val_inputs = input_scaler.transform(val_inputs)
    val_labels = target_scaler.transform(val_labels.reshape(-1, 1)).reshape(-1)

    test_inputs = input_scaler.transform(test_inputs)
    test_labels = target_scaler.transform(test_labels.reshape(-1, 1)).reshape(-1)

In [11]:
train_inputs = torch.from_numpy(train_inputs).float()
train_labels = torch.from_numpy(train_labels).float()

val_inputs = torch.from_numpy(val_inputs).float()
val_labels = torch.from_numpy(val_labels).float()

test_inputs = torch.from_numpy(test_inputs).float()
test_labels = torch.from_numpy(test_labels).float()

In [12]:
print ("Train size:", len(train_inputs))
print ("Val size:", len(val_inputs))
print ("Test size:", len(test_inputs))

Train size: 266784
Val size: 33348
Test size: 33348


In [13]:
BATCH_SIZE = 512

class metricsDataset (Dataset):
    def __init__(self, metrics, labels):
        self.metrics = metrics
        self.labels = labels

    def __len__(self):
        return len(self.metrics)

    def __getitem__(self, idx):
        return self.metrics[idx], self.labels[idx]

train_dataset = metricsDataset(train_inputs, train_labels)
val_dataset   = metricsDataset(val_inputs, val_labels)
test_dataset  = metricsDataset(test_inputs, test_labels)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    pin_memory=True
)

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

Using: cuda


In [15]:
model = nn.Sequential(
    nn.Linear(6, 128),
    nn.BatchNorm1d(128),
    nn.ReLU(),

    nn.Linear(128, 64),
    nn.BatchNorm1d(64),
    nn.ReLU(),

    nn.Linear(64, 1)
).to(device)

In [16]:
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=0.01
)

In [ ]:
EPOCHS = 160

best_val_loss = float("inf")

for epoch in range(0, EPOCHS):
    model.train()
    train_loss = 0.0

    for x, y in train_loader:
        optimizer.zero_grad()

        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        preds = model(x).squeeze(-1)
        loss = criterion(preds, y)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * x.size(0)

    train_loss /= len(train_loader.dataset)

    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            pred = model(x).squeeze(-1)
            loss = criterion(pred, y)

            val_loss += loss.item() * x.size(0)

    val_loss /= len(val_loader.dataset)

    print(
        f"Epoch {epoch + 1:3d}/{EPOCHS} | "
        f"Train MSE: {train_loss:.6f} | "
        f"Val MSE: {val_loss:.6f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss

        # SAVE the model
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_val_loss': best_val_loss,
        }, 'best_model_checkpoint-light.pth')

        print("Best Model Saved")

Epoch   1/160 | Train MSE: 0.372142 | Val MSE: 0.319022
Best Model Saved
Epoch   2/160 | Train MSE: 0.314159 | Val MSE: 0.314814
Best Model Saved
Epoch   3/160 | Train MSE: 0.302987 | Val MSE: 0.284103
Best Model Saved
Epoch   4/160 | Train MSE: 0.295141 | Val MSE: 0.303708
Epoch   5/160 | Train MSE: 0.285614 | Val MSE: 0.268277
Best Model Saved
Epoch   6/160 | Train MSE: 0.282266 | Val MSE: 0.270512
Epoch   7/160 | Train MSE: 0.274036 | Val MSE: 0.269416
Epoch   8/160 | Train MSE: 0.269407 | Val MSE: 0.269287
Epoch   9/160 | Train MSE: 0.265254 | Val MSE: 0.247625
Best Model Saved
Epoch  10/160 | Train MSE: 0.260793 | Val MSE: 0.259354
Epoch  11/160 | Train MSE: 0.257741 | Val MSE: 0.249414
Epoch  12/160 | Train MSE: 0.254823 | Val MSE: 0.295624
Epoch  13/160 | Train MSE: 0.251149 | Val MSE: 0.334322


In [ ]:
checkpoint = torch.load('best_model_checkpoint-light.pth')
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded best model (Val Loss: {checkpoint['best_val_loss']})")

model.eval()
test_loss = 0.0
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        pred = model(x).squeeze(-1)
        loss = criterion(pred, y)
        test_loss += loss.item() * x.size(0)

test_loss /= len(test_loader.dataset)
print(f"\nTest MSE: {test_loss:.6f}")
print(f"Test RMSE: {test_loss ** 0.5:.6f}")

In [ ]:
rel_test_loss = (test_loss**0.5)*100 / (test_labels.max()-test_labels.min())
print(str(float(rel_test_loss)) + "%")